modelo_base

In [1]:
import pandas as pd
import numpy as np

df= pd.read_csv('features.csv', parse_dates=['fecha_indice'])
print(df.shape)
print('filas por paciente (max):', df['person_id'].value_counts().max())


(5814, 12)
filas por paciente (max): 1


In [2]:
corte = df['fecha_indice'].quantile(0.8)

train = df[df['fecha_indice'] <  corte].copy()
test  = df[df['fecha_indice'] >= corte].copy()

pac_test = set(test['person_id'])
train = train[~train['person_id'].isin(pac_test)]

print('fecha de corte:', corte.date())
print('pacientes en ambos lados:', len(set(train['person_id']) & pac_test))
for nombre, d in [('train', train), ('test', test)]:
    print(nombre, len(d), 'filas |', d['muerte_90d'].sum(), 'eventos |',
          round(d['muerte_90d'].mean()*100, 2), '% prevalencia')

fecha de corte: 2023-03-09
pacientes en ambos lados: 0
train 4651 filas | 310 eventos | 6.67 % prevalencia
test 1163 filas | 58 eventos | 4.99 % prevalencia


### Actividad 4 Partición
Corte temporal en el percentil 80 de `fecha_indice` (2023-03-09). Una fila por paciente,
0 pacientes compartidos. Train: 4,651 (310 eventos, 6.67 %). Test: 1,163 (58 eventos, 4.99 %).
La prevalencia baja en el periodo reciente (58 eventos).
Al comparar AUPRC, su línea base es la prevalencia de cada conjunto.

In [3]:
test.to_csv('test_BLOQUEADO.csv', index=False)
X_cols = ['edad','sexo_masc','dm2_prev','ic_prev','erc_prev','demencia_prev','creatinina','hemoglobina']
X_train, y_train = train[X_cols], train['muerte_90d']
del test, df
# NOTA: El conjunto de prueba quedó guardado en test_BLOQUEADO.csv 

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_cols = ['edad', 'creatinina', 'hemoglobina']
bin_cols = ['sexo_masc', 'dm2_prev', 'ic_prev', 'erc_prev', 'demencia_prev']

prepro = ColumnTransformer([
    ('num', Pipeline([
        ('imputar', SimpleImputer(strategy='median', add_indicator=True)),
        ('escalar', StandardScaler())
    ]), num_cols),
    ('bin', SimpleImputer(strategy='most_frequent'), bin_cols)
])

def hacer_pipe(modelo):
    return Pipeline([('prepro', prepro), ('modelo', modelo)])

prepro

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

modelos = {
    'Dummy':     DummyClassifier(strategy='prior'),
    'Logistica': LogisticRegression(C=1.0, max_iter=1000),
    'Bosque':    RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                        random_state=2026, n_jobs=-1),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
grupos = train['person_id']

res = []
for nombre, m in modelos.items():
    r = cross_validate(hacer_pipe(m), X_train, y_train, groups=grupos, cv=cv,
                       scoring=['roc_auc', 'average_precision', 'precision', 'recall'])
    res.append({'modelo': nombre,
                'AUROC':  f"{r['test_roc_auc'].mean():.3f} ± {r['test_roc_auc'].std():.3f}",
                'AUPRC':  f"{r['test_average_precision'].mean():.3f} ± {r['test_average_precision'].std():.3f}",
                'Precision': f"{r['test_precision'].mean():.3f}",
                'Recall':    f"{r['test_recall'].mean():.3f}"})

print('Línea base AUPRC (prevalencia train):', round(y_train.mean(), 3))
pd.DataFrame(res)

/home/alecedillo/venv_go/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/alecedillo/venv_go/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/alecedillo/venv_go/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/alecedi

Línea base AUPRC (prevalencia train): 0.067


/home/alecedillo/venv_go/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,modelo,AUROC,AUPRC,Precision,Recall
0,Dummy,0.500 ± 0.000,0.067 ± 0.000,0.000,0.000
1,Logistica,0.733 ± 0.021,0.159 ± 0.022,0.000,0.000
2,Bosque,0.755 ± 0.036,0.226 ± 0.046,0.000,0.000


In [6]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
from sklearn.model_selection import cross_val_predict, KFold, cross_val_score
from sklearn.metrics import precision_score, recall_score

filas = []
for nombre in ['Logistica', 'Bosque']:
    p = cross_val_predict(hacer_pipe(modelos[nombre]), X_train, y_train,
                          groups=grupos, cv=cv, method='predict_proba')[:, 1]
    umbral = np.quantile(p, 0.90)
    pred = (p >= umbral).astype(int)
    filas.append({'modelo': nombre, 'umbral': round(umbral, 3),
                  'Precision': round(precision_score(y_train, pred), 3),
                  'Recall':    round(recall_score(y_train, pred), 3)})
display(pd.DataFrame(filas))

kf = KFold(n_splits=5, shuffle=True, random_state=2026)
for nombre in ['Logistica', 'Bosque']:
    a = cross_val_score(hacer_pipe(modelos[nombre]), X_train, y_train, cv=kf, scoring='roc_auc')
    print(nombre, 'KFold simple AUROC:', round(a.mean(), 3), '±', round(a.std(), 3))

,modelo,umbral,Precision,Recall
0,Logistica,0.157,0.155,0.232
1,Bosque,0.169,0.208,0.313


Logistica KFold simple AUROC: 0.735 ± 0.046
Bosque KFold simple AUROC: 0.752 ± 0.043


### Actividad 8 Esquema de validación
Uso StratifiedGroupKFold (5 pliegues) agrupando por `person_id`: los pacientes nunca
se reparten entre entrenamiento y validación, y cada pliegue conserva una prevalencia
parecida (6.7 %). Cada paciente tiene
una sola fila, así que KFold simple da casi lo mismo (logística 0.735 vs 0.733;
bosque 0.752 vs 0.755). La diferencia aparecería si un paciente tuviera varios
egresos: KFold pondría egresos del mismo paciente en ambos lados y el modelo
"reconocería" al paciente, inflando el AUC.

### Elección del modelo
El bosque supera a la logística en AUPRC (0.226 vs 0.159), precisión (0.21 vs 0.16)
y recall (0.31 vs 0.23), aunque en AUROC la diferencia (0.02) cae dentro de una
desviación estándar. Elijo el bosque para la prueba.

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score

# ACTIVIDAD 9 
test = pd.read_csv('test_BLOQUEADO.csv', parse_dates=['fecha_indice'])
X_test, y_test = test[X_cols], test['muerte_90d']

final = hacer_pipe(modelos['Bosque']).fit(X_train, y_train)
p_test = final.predict_proba(X_test)[:, 1]

umbral_cv = 0.169
pred_test = (p_test >= umbral_cv).astype(int)

print('Prevalencia test (línea base AUPRC):', round(y_test.mean(), 3))
print('AUROC test:', round(roc_auc_score(y_test, p_test), 3))
print('AUPRC test:', round(average_precision_score(y_test, p_test), 3))
print('Precision test:', round(precision_score(y_test, pred_test), 3))
print('Recall test:', round(recall_score(y_test, pred_test), 3))
print('Marcados alto riesgo:', pred_test.sum(), 'de', len(pred_test))

Prevalencia test (línea base AUPRC): 0.05
AUROC test: 0.851
AUPRC test: 0.217
Precision test: 0.28
Recall test: 0.362
Marcados alto riesgo: 75 de 1163


### Actividad 9 Resultado en prueba
Bosque aleatorio entrenado con todo train; umbral 0.169 fijado en validación.
Test: AUROC 0.851, AUPRC 0.217 (línea base 0.050), precisión 0.280, recall 0.362
(75 de 1,163 marcados).

Comparación: el AUROC en prueba (0.851) es mayor que en validación
cruzada (0.755 ± 0.036), lo que el genera desconfianza.

Explicaciones probables: (1) variabilidad por
el tamaño, ya que con 58 eventos el AUROC tiene un IC ~±0.05–0.06; (2) cambio
temporal, porque el periodo reciente tiene menor mortalidad (5.0 % vs 6.7 %) y
posiblemente otro patrón de registro en Synthea. No reajusté nada tras ver la
prueba. 

In [8]:
# ACTIVIDAD 10 
cols_fuga = X_cols + ['dias_seguimiento']
X_fuga = train[cols_fuga]

prepro_fuga = ColumnTransformer([
    ('num', Pipeline([('imputar', SimpleImputer(strategy='median', add_indicator=True)),
                      ('escalar', StandardScaler())]), num_cols + ['dias_seguimiento']),
    ('bin', SimpleImputer(strategy='most_frequent'), bin_cols)
])

tabla = []
for nombre in ['Logistica', 'Bosque']:
    pipe_ok   = hacer_pipe(modelos[nombre])
    pipe_fuga = Pipeline([('prepro', prepro_fuga), ('modelo', modelos[nombre])])
    ok   = cross_validate(pipe_ok,   X_train, y_train, groups=grupos, cv=cv,
                          scoring=['roc_auc', 'average_precision'])
    fuga = cross_validate(pipe_fuga, X_fuga,  y_train, groups=grupos, cv=cv,
                          scoring=['roc_auc', 'average_precision'])
    tabla.append({'modelo': nombre,
                  'AUROC sin fuga': round(ok['test_roc_auc'].mean(), 3),
                  'AUROC con fuga': round(fuga['test_roc_auc'].mean(), 3),
                  'AUPRC sin fuga': round(ok['test_average_precision'].mean(), 3),
                  'AUPRC con fuga': round(fuga['test_average_precision'].mean(), 3)})
pd.DataFrame(tabla)

,modelo,AUROC sin fuga,AUROC con fuga,AUPRC sin fuga,AUPRC con fuga
0,Logistica,0.733,0.995,0.159,0.994
1,Bosque,0.755,0.998,0.226,0.996


In [9]:
assert 'dias_seguimiento' not in X_cols, 'FUGA: variable post-índice en predictores'
print('Corrección verificada: predictores =', X_cols)

Corrección verificada: predictores = ['edad', 'sexo_masc', 'dm2_prev', 'ic_prev', 'erc_prev', 'demencia_prev', 'creatinina', 'hemoglobina']


In [10]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.statistics import proportional_hazard_test

def preparar(d, medianas):
    X = d[X_cols].copy()
    for col in ['creatinina', 'hemoglobina']:
        X[col + '_falta'] = X[col].isna().astype(int)
        X[col] = X[col].fillna(medianas[col])
    return X

In [11]:
med = train[['creatinina', 'hemoglobina']].median()
datos_cox = preparar(train, med)
datos_cox['T'] = train['dias_seguimiento']
datos_cox['E'] = train['muerte_90d']

cph = CoxPHFitter(penalizer=0.01)
cph.fit(datos_cox, duration_col='T', event_col='E')
cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].round(3)

,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,p
covariate,,,,
edad,1.031,1.014,1.048,0.000
sexo_masc,0.972,0.787,1.199,0.788
dm2_prev,1.120,0.869,1.442,0.381
ic_prev,3.244,2.580,4.080,0.000
erc_prev,0.867,0.688,1.094,0.228
demencia_prev,1.156,0.843,1.585,0.368
creatinina,1.037,1.005,1.071,0.024
hemoglobina,0.908,0.856,0.963,0.001
creatinina_falta,0.707,0.466,1.071,0.102


In [12]:
ph = proportional_hazard_test(cph, datos_cox, time_transform='rank')
ph.summary.round(3)

,test_statistic,p,-log2(p)
creatinina,0.016,0.898,0.155
creatinina_falta,14.373,0.000,12.704
demencia_prev,3.859,0.049,4.337
dm2_prev,1.491,0.222,2.171
edad,18.332,0.000,15.718
erc_prev,7.691,0.006,7.493
hemoglobina,5.094,0.024,5.380
hemoglobina_falta,0.016,0.900,0.152
ic_prev,25.372,0.000,21.012
sexo_masc,0.542,0.461,1.116


In [13]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

cv_cox = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
c_idx, auc = [], []
for i_tr, i_va in cv_cox.split(train, train['muerte_90d'], groups=train['person_id']):
    tr, va = train.iloc[i_tr], train.iloc[i_va]
    m = tr[['creatinina', 'hemoglobina']].median()
    d_tr = preparar(tr, m).assign(T=tr['dias_seguimiento'], E=tr['muerte_90d'])
    modelo = CoxPHFitter(penalizer=0.01).fit(d_tr, duration_col='T', event_col='E')
    riesgo = modelo.predict_partial_hazard(preparar(va, m))
    c_idx.append(concordance_index(va['dias_seguimiento'], -riesgo, va['muerte_90d']))
    auc.append(roc_auc_score(va['muerte_90d'], riesgo))

print('C-index CV:', round(np.mean(c_idx), 3), '±', round(np.std(c_idx), 3))
print('AUROC CV  :', round(np.mean(auc), 3), '±', round(np.std(auc), 3))

C-index CV: 0.724 ± 0.019
AUROC CV  : 0.733 ± 0.021


### Sección opcional. Regresión de Cox
Tiempo = `dias_seguimiento` (muerte o censura a 90 d), evento = `muerte_90d`. Solo entrenamiento;
imputación por mediana con indicador de faltante dentro de cada pliegue; penalizer = 0.01.
- HR principales: insuficiencia cardiaca 3.24 (2.58–4.08), edad 1.031/año, hemoglobina 0.908/g·dL,
  creatinina 1.037/mg·dL. DM2, ERC, demencia y sexo sin asociación independiente.
- Hemoglobina faltante HR 0.43: probable sesgo de registro (menos contacto con el sistema), no causal.
- Supuesto de riesgos proporcionales violado (p < 0.05) en edad, IC, ERC, hemoglobina y creatinina_falta;
  la prueba es sensible con n grande. Pendiente: residuos de Schoenfeld / estratificación.
- CV (mismos pliegues de la act. 7): C-index 0.724 ± 0.019; AUROC 0.733 ± 0.021, igual que la
  logística. El conjunto de prueba no se reabrió.